In [1]:
import pandas as pd
import numpy as np
import kagglehub

# 1. Load Data
print("Downloading and loading data...")
path = kagglehub.dataset_download("jeandedieunyandwi/lending-club-dataset")
df = pd.read_csv(path + "/lending_club_loan_two.csv")

100%|██████████| 27.6M/27.6M [00:01<00:00, 21.1MB/s]

Extracting files...


In [4]:
# 2. Define Target Variable

In [3]:
df['default'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)

In [5]:
# 3. Handle Missing Values
print("Cleaning missing values...")
df['emp_length'] = df['emp_length'].fillna('Unknown')
df['revol_util'] = df['revol_util'].fillna(df['revol_util'].median())
df['pub_rec_bankruptcies'] = df['pub_rec_bankruptcies'].fillna(0)
df['mort_acc'] = df['mort_acc'].fillna(0)

Cleaning missing values...


In [6]:
# 4. Feature Engineering
print("Engineering new features...")
# Extract Zip Code
df['zip_code'] = df['address'].apply(lambda address: address[-5:])
# Convert Term to Integer
df['term'] = df['term'].apply(lambda term: int(term[:3]))
# Extract Earliest Credit Year
df['earliest_cr_year'] = pd.to_datetime(df['earliest_cr_line']).dt.year
# Fix Income Skewness
df['log_income'] = np.log1p(df['annual_inc'])

Engineering new features...


/tmp/ipykernel_2199/1377775308.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['earliest_cr_year'] = pd.to_datetime(df['earliest_cr_line']).dt.year


In [7]:
# 5. Group Categorical Variables
# Home Ownership
df['home_ownership'] = df['home_ownership'].apply(lambda x: x if x in ['MORTGAGE', 'OWN', 'RENT'] else 'OTHER')

In [8]:
# Purpose
def group_purpose(x):
    if x in ['small_business', 'medical', 'moving', 'other']:
        return 'high_risk'
    elif x in ['credit_card', 'debt_consolidation']:
        return 'medium_risk'
    else:
        return 'low_risk'
df['purpose_group'] = df['purpose'].apply(group_purpose)

In [9]:
# 6. Drop Unnecessary Columns to Prevent Data Leakage/Noise
cols_to_drop = [
    'grade', 'emp_length', 'issue_d', 'open_acc', 'total_acc',
    'address', 'earliest_cr_line', 'purpose', 'loan_status',
    'initial_list_status', 'application_type', 'emp_title', 'title'
]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [10]:
# 7. Export Cleaned Data
df.to_csv('lending_data_cleaned.csv', index=False)
print(f"✅ Data cleaned! Final shape: {df.shape}. Saved as 'lending_data_cleaned.csv'")

✅ Data cleaned! Final shape: (396030, 19). Saved as 'lending_data_cleaned.csv'
